# LangGraph + CoordiNode: Agent with graph memory

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/03_langgraph_agent.ipynb)

Demonstrates a LangGraph agent that uses CoordiNode as persistent **graph memory**:
- `save_fact` — store a subject → relation → object triple in the graph
- `query_facts` — run an arbitrary Cypher query against the graph
- `find_related` — traverse the graph from a given entity
- `list_all_facts` — dump every fact in the current session

**Works without OpenAI** — the mock demo section calls tools directly.  
Set `OPENAI_API_KEY` to run the full `gpt-4o-mini` ReAct agent.

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). First run compiles from source (~5 min); subsequent runs use the pip cache.
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

## Install dependencies

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# Install coordinode-embedded in Colab only (requires Rust build).
if IN_COLAB:
    # Install Rust toolchain via rustup (https://rustup.rs).
    # Colab's apt packages ship rustc ≤1.75, which cannot build coordinode-embedded
    # (requires Rust ≥1.80 for maturin/pyo3). apt-get is not a viable alternative here.
    # Download the installer to a temp file and execute it explicitly — this avoids
    # piping remote content directly into a shell while maintaining HTTPS/TLS security
    # through Python's default ssl context (cert-verified, TLS 1.2+).
    import ssl as _ssl, tempfile as _tmp, urllib.request as _ur

    _ctx = _ssl.create_default_context()
    with _tmp.NamedTemporaryFile(mode="wb", suffix=".sh", delete=False) as _f:
        with _ur.urlopen("https://sh.rustup.rs", context=_ctx) as _r:
            _f.write(_r.read())
        _rustup_path = _f.name
    try:
        subprocess.run(["/bin/sh", _rustup_path, "-s", "--", "-y", "-q"], check=True)
    finally:
        os.unlink(_rustup_path)
    # Add cargo to PATH so maturin/pip can find it.
    _cargo_bin = os.path.expanduser("~/.cargo/bin")
    os.environ["PATH"] = f"{_cargo_bin}{os.pathsep}{os.environ.get('PATH', '')}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/structured-world/coordinode-python.git#subdirectory=coordinode-embedded",
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "coordinode",
        "langchain-coordinode",
        "langchain-community",
        "langchain-openai",
        "langgraph",
    ],
    check=True,
)

print("SDK installed")

## Connect to CoordiNode

This notebook uses `.cypher()` directly, which is identical on both `LocalClient`
(embedded) and `CoordinodeClient` (gRPC). No adapter needed.

In [ ]:
import os, socket


def _port_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False


GRPC_PORT = int(os.environ.get("COORDINODE_PORT", "7080"))

if os.environ.get("COORDINODE_ADDR"):
    COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
elif _port_open(GRPC_PORT):
    COORDINODE_ADDR = f"localhost:{GRPC_PORT}"
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
else:
    # No server available — use the embedded in-process engine.
    from coordinode_embedded import LocalClient

    client = LocalClient(":memory:")
    print("Using embedded LocalClient (in-process)")

## 1. Define LangChain tools

Each tool maps a natural-language action to a Cypher operation.
A `SESSION` UUID isolates this demo's data from other concurrent runs.

In [ ]:
import os, re, uuid
from langchain_core.tools import tool

SESSION = uuid.uuid4().hex[:8]  # isolates this demo's data from other sessions

_REL_TYPE_RE = re.compile(r"[A-Z_][A-Z0-9_]*")
# Regex guards for query_facts (demo safety guard).
_WRITE_CLAUSE_RE = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|CALL|LOAD)\b",
    re.IGNORECASE,
)
# NOTE: this guard checks that AT LEAST ONE node pattern carries session scope.
# A Cartesian-product query such as `MATCH (n), (m {session: $sess}) RETURN n`
# would pass yet return unscoped rows for `n`.  A complete per-alias check would
# require parsing the Cypher AST, which is out of scope for a demo safety guard.
# In production code, use server-side row-level security instead of client regex.
_SESSION_SCOPE_RE = re.compile(
    r"WHERE\b[^;{}]*\.session\s*=\s*\$sess|\{[^}]*session\s*:\s*\$sess[^}]*\}",
    re.IGNORECASE,
)


@tool
def save_fact(subject: str, relation: str, obj: str) -> str:
    """Save a fact (subject → relation → object) into the knowledge graph.
    Example: save_fact('Alice', 'WORKS_AT', 'Acme Corp')"""
    rel_type = relation.upper().replace(" ", "_")
    # Validate rel_type before interpolating into Cypher to prevent injection.
    if not _REL_TYPE_RE.fullmatch(rel_type):
        return f"Invalid relation type {relation!r}: only letters, digits, and underscores allowed"
    client.cypher(
        f"MERGE (a:Entity {{name: $s, session: $sess}}) "
        f"MERGE (b:Entity {{name: $o, session: $sess}}) "
        f"MERGE (a)-[r:{rel_type}]->(b)",
        params={"s": subject, "o": obj, "sess": SESSION},
    )
    return f"Saved: {subject} -[{rel_type}]-> {obj}"


@tool
def query_facts(cypher: str) -> str:
    """Run a read-only Cypher MATCH query against the knowledge graph.
    Must scope reads via either WHERE <alias>.session = $sess
    or a node pattern {session: $sess}."""
    q = cypher.strip()
    if _WRITE_CLAUSE_RE.search(q):
        return "Only read-only Cypher is allowed in query_facts."
    # Require $sess in a WHERE clause or node pattern, not just anywhere.
    # Accepts both: WHERE n.session = $sess  and  MATCH (n {session: $sess})
    if not _SESSION_SCOPE_RE.search(q):
        return "Query must scope reads to the current session with either WHERE <alias>.session = $sess or {session: $sess}"
    rows = client.cypher(q, params={"sess": SESSION})
    return str(rows[:20]) if rows else "No results"


@tool
def find_related(entity_name: str, depth: int = 1) -> str:
    """Find all entities reachable from entity_name within the given number of hops (max 3)."""
    safe_depth = max(1, min(int(depth), 3))
    rows = client.cypher(
        f"MATCH (n:Entity {{name: $name, session: $sess}})-[r*1..{safe_depth}]->(m:Entity {{session: $sess}}) "
        "RETURN m.name AS related, type(last(r)) AS via LIMIT 20",
        params={"name": entity_name, "sess": SESSION},
    )
    if not rows:
        return f"No related entities found for {entity_name}"
    return "\n".join(f"{r['via']} -> {r['related']}" for r in rows)


@tool
def list_all_facts() -> str:
    """List every fact stored in the current session's knowledge graph."""
    rows = client.cypher(
        "MATCH (a:Entity {session: $sess})-[r]->(b:Entity {session: $sess}) "
        "RETURN a.name AS subject, type(r) AS relation, b.name AS object",
        params={"sess": SESSION},
    )
    if not rows:
        return "No facts stored yet"
    return "\n".join(f"{r['subject']} -[{r['relation']}]-> {r['object']}" for r in rows)


tools = [save_fact, query_facts, find_related, list_all_facts]
print(f"Session: {SESSION}")
print("Tools:", [t.name for t in tools])

## 2. Mock demo — no LLM required (direct tool calls)

Shows the full graph memory workflow by calling the tools directly.

In [ ]:
print("=== Saving facts ===")
print(save_fact.invoke({"subject": "Alice", "relation": "WORKS_AT", "obj": "Acme Corp"}))
print(save_fact.invoke({"subject": "Alice", "relation": "MANAGES", "obj": "Bob"}))
print(save_fact.invoke({"subject": "Bob", "relation": "WORKS_AT", "obj": "Acme Corp"}))
print(save_fact.invoke({"subject": "Acme Corp", "relation": "LOCATED_IN", "obj": "Berlin"}))
print(save_fact.invoke({"subject": "Alice", "relation": "KNOWS", "obj": "Charlie"}))
print(save_fact.invoke({"subject": "Charlie", "relation": "EXPERT_IN", "obj": "Machine Learning"}))

print("\n=== All facts in session ===")
print(list_all_facts.invoke({}))

print("\n=== Related to Alice (depth=1) ===")
print(find_related.invoke({"entity_name": "Alice", "depth": 1}))

print("\n=== Related to Alice (depth=2) ===")
print(find_related.invoke({"entity_name": "Alice", "depth": 2}))

print("\n=== Cypher query: who works at Acme Corp? ===")
print(
    query_facts.invoke(
        {
            "cypher": 'MATCH (p:Entity {session: $sess})-[:WORKS_AT]->(c:Entity {name: "Acme Corp"}) RETURN p.name AS employee'
        }
    )
)

## 3. LangGraph StatefulGraph — manual workflow

Shows how to wire CoordiNode tool calls into a LangGraph state machine without an LLM.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator


class AgentState(TypedDict):
    messages: Annotated[list[str], operator.add]
    facts_saved: int


def extract_and_save_node(state: AgentState) -> AgentState:
    """Simulates entity extraction: saves a hardcoded fact.
    In production this node would call an LLM to extract entities from the last message."""
    result = save_fact.invoke({"subject": "DemoSubject", "relation": "DEMO_REL", "obj": "DemoObject"})
    return {"messages": [f"[extract] {result}"], "facts_saved": state["facts_saved"] + 1}


def query_node(state: AgentState) -> AgentState:
    """Reads the graph and appends a summary to messages."""
    result = list_all_facts.invoke({})
    return {"messages": [f"[query] Facts: {result[:200]}"], "facts_saved": state["facts_saved"]}


def should_query(state: AgentState) -> str:
    return "query" if state["facts_saved"] >= 1 else "extract"


builder = StateGraph(AgentState)
builder.add_node("extract", extract_and_save_node)
builder.add_node("query", query_node)
builder.set_entry_point("extract")
builder.add_conditional_edges("extract", should_query, {"query": "query", "extract": "extract"})
builder.add_edge("query", END)

graph_agent = builder.compile()

result = graph_agent.invoke({"messages": ["Tell me about Alice"], "facts_saved": 0})
print("Graph agent output:")
for msg in result["messages"]:
    print(" ", msg)

## 4. LangGraph ReAct agent (optional — requires OPENAI_API_KEY)

> Set `OPENAI_API_KEY` in your environment or via
> `os.environ['OPENAI_API_KEY'] = 'sk-...'` before running.
> The cell is skipped automatically when the key is absent.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping LLM agent. See section 2 for the mock demo.")
else:
    from langchain_openai import ChatOpenAI
    from langgraph.prebuilt import create_react_agent

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_react_agent(llm, tools)

    print("Agent ready. Running demo conversation...")
    messages = [
        {
            "role": "user",
            "content": "Save these facts: Alice works at Acme Corp, Alice manages Bob, Acme Corp is in Berlin.",
        },
        {"role": "user", "content": "Who does Alice manage?"},
        {"role": "user", "content": "What are all the facts about Alice?"},
    ]
    for msg in messages:
        print(f"\n>>> {msg['content']}")
        result = agent.invoke({"messages": [msg]})
        print("Agent:", result["messages"][-1].content)

## Cleanup

In [ ]:
client.cypher("MATCH (n:Entity {session: $sess}) DETACH DELETE n", params={"sess": SESSION})
print("Cleaned up session:", SESSION)
client.close()